# QCoDeS example with Zurich Instruments MFLI and UHFLI

This notebook shows how to connect to either lock-in, set individual parameters,
read X, Y, R and Theta, and save point measurements and time traces with QCoDeS.
It uses the base instruments, without licensed options. All channel indices are
zero-based: `demods[0]` is demodulator 1 in LabOne.

## Connect to the instrument

1. Set up QCoDeS (see [14 minutes to QCoDeS](https://github.com/lairdgrouplancaster/14-minutes-to-QCoDeS/blob/main/14_minutes_to_QCoDeS.ipynb)).
   In this repository's Python environment, install the working drivers with
   `python -m pip install -e .`, and install `zhinst-qcodes` and `matplotlib`.
   Use a `zhinst-core` version compatible with your installed LabOne server.
2. Connect the instrument by Ethernet or USB and start the LabOne data server.
   Check that LabOne can see its serial number. These drivers connect through
   that server; `localhost` is the computer running LabOne, not the instrument's IP address.
3. Choose `MODEL` below. The example uses MFLI **dev7920 over Ethernet** or
   UHFLI **dev20046 over USB**. Change the serial, host, port or interface for your setup.
4. For a signal test, connect Signal Output 1 to Signal Input 1 with a suitable
   cable. Without the cable, the notebook can still acquire input noise.

Run the cells in order, using a fresh kernel or closing the previous instrument
before repeating the connection cell. The example leaves signal generation off
unless you explicitly change `ENABLE_OUTPUT` in the setup cell.

In [ ]:
from qcodes_contrib_drivers.drivers.ZurichInstruments.MFLI import MFLI
from qcodes_contrib_drivers.drivers.ZurichInstruments.UHFLI import UHFLI

MODEL = "MFLI"  # Change to "UHFLI" to run the same example on that instrument.
HOST = "localhost"

if MODEL == "MFLI":
    lia = MFLI("mfli", "dev7920", HOST, port=8004, interface="1GbE")
elif MODEL == "UHFLI":
    lia = UHFLI("uhfli", "dev20046", HOST, port=8004, interface="USB")
else:
    raise ValueError("MODEL must be MFLI or UHFLI")

lia.get_idn()

## Initialise QCoDeS control

A station collects the instrument configuration for dataset snapshots. The
database and experiment below hold all the measurements made in this notebook.
`lia` is the QCoDeS instrument; the shorter names refer to its existing channels.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from qcodes import Station
from qcodes.dataset import (
    do0d, do1d, do2d,
    initialise_or_create_database_at,
    load_or_create_experiment,
)
from qcodes.parameters import MultiParameter

station = Station(lia)
data_directory = Path("example_output")
data_directory.mkdir(exist_ok=True)
initialise_or_create_database_at(str(data_directory / "zurich_lockins.db"))
experiment = load_or_create_experiment(
    experiment_name=f"testing_{MODEL.lower()}", sample_name="signal_source"
)

demod_index = 0
demod = lia.demods[demod_index]
oscillator = lia.oscs[0]
signal_input = lia.sigins[0]
signal_output = lia.sigouts[0]

## Set and read individual parameters

Call a parameter with a value to set it, and without a value to read it:
`oscillator.freq(10e3)` sets the frequency; `oscillator.freq()` reads it back.

| Quantity | Parameter | Meaning |
| --- | --- | --- |
| Frequency | `oscillator.freq` | Oscillator frequency in Hz |
| Amplitude | `signal_output.amplitude` | Peak volts at the configured output load |
| Phase | `demod.phaseshift` | Reference phase shift in degrees |
| Analog input gain | `signal_input.range` | Input range in volts; selecting the range sets the front-end sensitivity |
| Digital gain/scaling | `signal_input.scaling` | Multiplier already applied to demodulator values |
| Low-pass filter | `demod.timeconstant`, `demod.order` | Time constant in seconds and filter order |
| Host streaming | `demod.rate`, `demod.enable` | Requested samples/s and streaming enable |

There is no generic `gain` parameter on these drivers. Choose an input range
that accommodates the signal, then use `scaling` only for a deliberate digital
gain or calibration. Read back settings because the hardware can quantize them.

The cell below selects Signal Input 1, an internal reference, and a 10 kHz,
10 mV peak excitation. `amplitude_enabled` selects the oscillator contribution;
`on` controls the physical output. The amplitude alias maps to mixer 1 on the
MFLI and mixer 3 on UHFLI output 1 (mixer 7 on UHFLI output 2).
The output mixer has its own reference settings. The setup also selects
oscillator 0, harmonic 1 and zero phase for that mixer, following the
[vendor signal-generation example](https://docs.zhinst.com/zhinst-toolkit/en/latest/examples/scope_module.html).
The measured phase is then controlled independently by `demod.phaseshift`.


In [ ]:
ENABLE_OUTPUT = False  # Set True after connecting the loopback cable or your DUT.

signal_output.on(0)
lia.extrefs[0].enable(0)  # Internal reference for oscillator 0.
signal_input.on(1)
signal_input.diff(0)     # Single-ended Signal Input 1.
signal_input.ac(0)       # DC coupling.
signal_input.imp50(0)    # High-impedance input.
signal_input.range(0.3)  # Analog input range, V.
signal_input.scaling(1.0)

signal_output.imp50(0)   # Amplitude is specified for a high-impedance load.
signal_output.range(1.0)
signal_output.offset(0.0)
oscillator.freq(10e3)
output_mixer = 1 if MODEL == "MFLI" else 3
output_reference = lia.demods[output_mixer]
output_reference.oscselect(0)
output_reference.harmonic(1)
output_reference.phaseshift(0.0)
signal_output.amplitude(0.010)
signal_output.amplitude_enabled(1)

demod.adcselect(0)       # Signal Input 1 on either model.
demod.oscselect(0)
demod.harmonic(1)
demod.phaseshift(0.0)
demod.order(3)
demod.timeconstant(0.010)
demod.sinc(0)
demod.trigger(0)         # Continuous streaming.
demod.rate(1000)
demod.enable(1)
lia.configure_readout(demod=demod_index)  # Infer V for this unscaled voltage input.
signal_output.on(int(ENABLE_OUTPUT))

# Allow this example's filter to settle before the first reading.
settle_s = max(0.2, 20 * float(demod.timeconstant()))
time.sleep(settle_s)

print("Frequency:", oscillator.freq(), "Hz")
print("Amplitude:", signal_output.amplitude(), "V peak")
print("Phase:", demod.phaseshift(), "deg")
print("Input range:", signal_input.range(), "V")
print("Digital scaling:", signal_input.scaling())
print("Filter:", demod.order(), "poles;", demod.timeconstant(), "s")
print("Actual streaming rate:", demod.rate(), "samples/s")

### Digital gain and units

For example, apply a digital gain of ten and label the values accordingly.
`configure_readout` changes QCoDeS metadata; it does not multiply the values
again. Refresh this metadata **before starting a new dataset** after changing
source or scaling. Reads reject stale source/scaling metadata.

This cell returns to unit scaling for all the following examples. For an
unscaled MFLI current input (`adcselect=1`), unit inference instead gives amps;
UHFLI `adcselect=1` selects its second voltage input.

In [ ]:
try:
    signal_input.scaling(10.0)
    lia.configure_readout(demod=demod_index, unit="scaled V")
    time.sleep(settle_s)
    print("Scaled R:", demod.r(), demod.r.unit)
finally:
    signal_input.scaling(1.0)
    lia.configure_readout(demod=demod_index)
time.sleep(settle_s)

## Read one point of X, Y, R and Theta

The individual parameters return Python floats. X, Y and R use RMS volts for
the unscaled voltage input configured above; Theta is in degrees.

In [ ]:
x = demod.x()
y = demod.y()
r = demod.r()
theta = demod.theta()
print(f"X={x:g} V, Y={y:g} V, R={r:g} V, Theta={theta:g} deg")

Those four calls acquire independently. To obtain all four quantities from
**one sample**, use `readout`. This is a QCoDeS `MultiParameter`, so it can be
passed directly to `do0d`, `do1d` or `do2d`. R is `hypot(X, Y)` and Theta is
`degrees(atan2(Y, X))`, between -180 and +180 degrees.

Each new point waits for the device timestamp to advance beyond an initial
sample. This does not guarantee filter or experiment settling: choose the
sweep delay accordingly. `lia.acquisition_timeout(5)` sets the local timeout
in seconds; it cannot interrupt a blocking LabOne transport call.

In [ ]:
x, y, r, theta = demod.readout()
print(f"Coherent reading: X={x:g}, Y={y:g}, R={r:g}, Theta={theta:g}")

## Point-by-point acquisition using do1d

Sweep frequency, wait for settling, then acquire one coherent X/Y/R/Theta
sample at each frequency. The dataset contains four separate result columns;
the plots include X and Y versus frequency.

If only independent X and Y readings are needed, replace `demod.readout` with
`demod.x, demod.y`. These do not share a sample. With a direct loopback cable
a flat amplitude response is expected over this small frequency range.

In [ ]:
frequency_start = oscillator.freq()
try:
    points_vs_frequency, _, _ = do1d(
        oscillator.freq, 8e3, 12e3, 11, settle_s,
        demod.readout,
        exp=experiment,
        measurement_name="points_vs_frequency",
        do_plot=True,
        show_progress=True,
        use_threads=False,
    )
finally:
    oscillator.freq(frequency_start)

## Point-by-point acquisition using do2d

Sweep output amplitude in the outer loop and frequency in the inner loop.
`do2d` saves one X/Y/R/Theta point for every amplitude-frequency pair, giving
two-dimensional maps. The first parameter is the slow axis. For an experiment,
the outer parameter could instead be a gate voltage or magnetic field.

In [ ]:
amplitude_start = signal_output.amplitude()
frequency_start = oscillator.freq()
try:
    points_vs_amplitude_frequency, _, _ = do2d(
        signal_output.amplitude, 0.005, 0.015, 3, settle_s,
        oscillator.freq, 8e3, 12e3, 11, settle_s,
        demod.readout,
        exp=experiment,
        measurement_name="points_vs_amplitude_frequency",
        do_plot=True,
        show_progress=True,
        use_threads=False,
    )
finally:
    signal_output.amplitude(amplitude_start)
    oscillator.freq(frequency_start)

## Acquire an X/Y trace versus time

A trace uses the inherited **LabOne DAQ module** to record many samples in
one acquisition. Calling the scalar `x()` parameter repeatedly would instead
make separate point measurements with timing determined partly by Python.

The helper below is notebook code, not an additional driver API. It wraps one
DAQ burst as a QCoDeS `MultiParameter` with two arrays and a shared time axis.
This lets `do0d` and `do1d` save both X and Y with their time setpoints.
It is deliberately configured for the unscaled Signal Input 1 used above.

Each call creates a fresh DAQ module, acquires one continuous burst, and
releases the module even if acquisition fails. Linear grid mode puts X and Y
on the same grid. The requested grid rate is kept below the actual demodulator
streaming rate. Time is calculated from the returned device timestamps using
`lia.clockbase()` (60 MHz on this MFLI, 1.8 GHz on this UHFLI), not Python timing.
A new trace starts its relative time axis at zero.

In [ ]:
class XYTrace(MultiParameter):
    """One DAQ burst of X and Y, for this notebook's voltage-input setup."""

    def __init__(self, name, *, instrument, demod_index, duration=0.5,
                 points=201, timeout=10.0, **kwargs):
        self.device = instrument.root_instrument
        self.demod = instrument
        self.demod_index = demod_index
        self.duration = float(duration)
        self.points = int(points)
        self.timeout = float(timeout)
        if (not np.isfinite([self.duration, self.timeout]).all()
                or self.duration <= 0 or self.points < 2 or self.timeout <= 0):
            raise ValueError("Use positive duration/timeout and at least two points")
        self.time_axis = np.arange(self.points) * self.duration / self.points
        super().__init__(
            name, instrument=instrument,
            names=("trace_x", "trace_y"), labels=("X", "Y"), units=("V", "V"),
            shapes=((self.points,), (self.points,)),
            setpoints=((self.time_axis,), (self.time_axis,)),
            setpoint_names=(("trace_time",), ("trace_time",)),
            setpoint_labels=(("Time from trace start",),) * 2,
            setpoint_units=(("s",),) * 2,
            snapshot_get=False, snapshot_value=False, **kwargs,
        )
        self.metadata.update(duration_s=self.duration, points=self.points,
                             grid_mode="linear", demodulator=demod_index)

    def _check_setup(self):
        if self.demod.adcselect() != 0 or self.device.sigins[0].scaling() != 1:
            raise RuntimeError("This example trace requires unscaled Signal Input 1")
        if not self.demod.enable() or self.demod.trigger() != 0:
            raise RuntimeError("Enable continuous demodulator streaming first")
        if any(ref.enable() and ref.demodselect() == self.demod_index
               for ref in self.device.extrefs):
            raise RuntimeError("The demodulator is reserved for external reference")
        if self.points / self.duration > self.demod.rate():
            raise ValueError("Increase streaming rate or reduce points/duration")

    def get_raw(self):
        self._check_setup()
        paths = [f"/{self.device.serial}/demods/{self.demod_index}/sample.{c}"
                 for c in ("x", "y")]
        module = self.device.session.modules.create_daq_module()
        try:
            module.device(self.device)
            module.type(0)          # Continuous acquisition, no trigger.
            module.grid.mode(2)     # Linear interpolation to a common grid.
            module.grid.rows(1)
            module.count(1)         # One burst per parameter read.
            module.duration(self.duration)
            module.grid.cols(self.points)
            for path in paths:
                module.subscribe(path)
            module.execute()
            module.wait_done(timeout=self.timeout, sleep_time=0.05)
            data = module.read(raw=True)
        finally:
            try:
                module.finish()
                module.unsubscribe("*")
            finally:
                try:
                    module.raw_module.clear()
                finally:
                    module.close()

        if any(path not in data or len(data[path]) != 1 for path in paths):
            raise RuntimeError("Expected one complete burst for each component")
        bursts = [data[path][0] for path in paths]
        values = [np.asarray(burst["value"]) for burst in bursts]
        stamps = [np.asarray(burst["timestamp"]) for burst in bursts]
        if any(a.shape != (1, self.points) for a in values + stamps):
            raise RuntimeError("The DAQ returned an incomplete trace")
        if not all(np.isfinite(a).all() for a in values):
            raise RuntimeError("The DAQ returned non-finite values")
        if not np.array_equal(stamps[0], stamps[1]):
            raise RuntimeError("X and Y timestamps do not match")
        ticks = stamps[0][0]
        if np.any(ticks[1:] <= ticks[:-1]):
            raise RuntimeError("Trace timestamps do not increase")
        times = (ticks - ticks[0]) / float(self.device.clockbase())
        self._check_setup()
        # DataSaver reads these setpoints after get_raw returns, so it saves
        # the actual returned time axis alongside this particular burst.
        self.time_axis = times
        self.setpoints = ((times,), (times,))
        return values[0][0], values[1][0]

In [ ]:
demod.add_parameter(
    "xy_trace", parameter_class=XYTrace, demod_index=demod_index,
    duration=0.5, points=201, timeout=10.0,
)
trace = demod.xy_trace
time.sleep(settle_s)
trace_x, trace_y = trace()  # Acquires a new pair of traces.
trace_time = trace.time_axis.copy()

fig, ax = plt.subplots()
ax.plot(trace_time, trace_x, label="X")
ax.plot(trace_time, trace_y, label="Y")
ax.set(xlabel="Time from trace start (s)", ylabel="Voltage (V)")
ax.legend()
plt.show()

The trace duration is 0.5 s with 201 grid points. The last point is just before
0.5 s, because duration includes the final sample interval. This is a
demodulated, low-pass-filtered trace, not the raw high-speed scope waveform.
R and Theta can also be calculated from the aligned X/Y arrays:

In [ ]:
trace_r = np.hypot(trace_x, trace_y)
trace_theta = np.degrees(np.arctan2(trace_y, trace_x))

## Trace-by-trace acquisition using do0d

`do0d` means **no swept parameter**. The measured parameter can still return
arrays: here it acquires a fresh X/Y trace and saves it against time. It does
not save the previous `trace_x` and `trace_y` arrays from the manual example.

In [ ]:
time.sleep(settle_s)
trace_at_one_frequency, _, _ = do0d(
    trace,
    exp=experiment,
    measurement_name="xy_vs_time",
    do_plot=True,
    use_threads=False,
)

## Trace-by-trace acquisition using do1d

Sweep frequency and record one fresh time trace at each step. There is one
swept parameter, so this uses `do1d`; each returned trace already has its own
time axis. The resulting X and Y datasets are two-dimensional: **frequency
and time since the start of each trace**.

Traces at different frequencies are separate acquisitions, with gaps for
setting, settling and DAQ setup. This is not one uninterrupted time stream.
Keep trace length and units fixed throughout a dataset. To use another
duration/length, create another trace parameter before starting a new dataset.

In [ ]:
frequency_start = oscillator.freq()
try:
    traces_vs_frequency, _, _ = do1d(
        oscillator.freq, 8e3, 12e3, 5, settle_s,
        trace,
        exp=experiment,
        measurement_name="xy_vs_time_and_frequency",
        do_plot=True,
        show_progress=True,
        use_threads=False,
    )
finally:
    oscillator.freq(frequency_start)

# Each result carries both its frequency and its time coordinates.
trace_data = traces_vs_frequency.get_parameter_data()
for name in trace.full_names:
    print(name, {column: array.shape for column, array in trace_data[name].items()})

## Close the connection

Each trace has already released its own DAQ module. Turn off the excitation
and streaming, remove this instrument from the station, detach it from the
data server, and close its QCoDeS object. Detaching affects access to this
device through the shared data server; do not detach a device another client
is still using. `lia.close()` alone only releases the QCoDeS object.

A vendor session may be shared with other Zurich instruments, so this cell
disconnects only the device selected above. Closing a QCoDeS object by itself
does not switch hardware outputs off or restore earlier device settings.
LabOne detaches the device asynchronously. Allow it to finish before immediately
reconnecting; an early reconnect can temporarily report `DeviceInUseError` or
`DeviceNotFoundError`.


In [ ]:
signal_output.on(0)
demod.enable(0)
session, serial = lia.session, lia.serial
station.remove_component(lia.name)
try:
    session.disconnect_device(serial)
finally:
    lia.close()

## Further controls and compatibility

The inherited node hierarchy also exposes auxiliary I/O, digital I/O, scope,
external-reference, system, MFLI threshold-unit and UHFLI arithmetic-unit
controls. The base MFLI has one measurement demodulator (`demods[0]`);
`demods[1]` is reference-only. The UHFLI has eight measurement demodulators;
channels assigned to active external-reference recovery cannot be read as
measurement channels. New reads report disabled or reserved channels rather
than enabling them automatically.

`demod.sample()` remains the vendor's raw dictionary. The existing
`demod.complex_sample()` returns the latest X + 1jY without waiting for a
newer timestamp and keeps its historical metadata. Neither acquisition helper
is read during snapshots. Scalars and `readout` are not synchronized across
different demodulators or instruments.

Scope, FFT, sweeper and device-settings modules remain available through
`lia.session.modules` while connected. Prefer `create_*_module()` for separate
jobs; named module properties may be cached and shared. For formatted DAQ
results use `module.read(clk_rate=lia.clockbase())`; its default 60 MHz clock
is incorrect for UHFLI timestamps. The helper above instead converts raw ticks
itself. No licensed-option features are used here.

See the [hardware validation notebook](ZurichInstruments_lockins_validation.ipynb)
for tested versions, the MFLI auxiliary-selector issue and remaining cable
checks. Further references:

- [QCoDeS doNd examples](https://microsoft.github.io/Qcodes/examples/DataSet/Using_doNd_functions_in_comparison_to_Measurement_context_manager_for_performing_measurements.html)
- [LabOne DAQ module](https://docs.zhinst.com/labone_api_user_manual/modules/daq/index.html)